# Step 2. Read the measures: their text and their cost

Each budget chapter is a page of headings and text, and ends with a **cost table**: one row per measure (millions of
dollars over the next five or six years), grouped by section, ending with the chapter's *net fiscal impact*. This
notebook turns the pages saved in step 1 into one list of measures, each with its text, its word count and its cost.

**What a measure is.** A heading and the text under it, down to the next heading of the same level:
- the measure level is **h3** in Budgets 2022 to 2026 and **h4** in Budget 2021 (which uses h3 for its numbered
  sections);
- any heading that has **its own row in the cost table** is a measure too, wherever it sits (e.g. Budget 2025 splits
  "Protecting Workers and Transforming Canada's Strategic Industries" into five costed parts, each an h4: each part is
  its own measure, so each can go to its own category);
- sub-headings without a row of their own, and call-out boxes (examples, project profiles, "past investments"), are
  part of the measure they sit in;
- left out: section introductions (and boxes placed before a section's first measure), impact reports, "key ongoing
  actions", "what we heard" and "investments since 2015" boxes, charts, tables, and headings with almost no text under
  them (fewer than 30 words and no cost).

**What a measure costs.** The net total of its rows in the chapter's cost table:
- a measure's "Less: funds previously provisioned / sourced from existing resources / projected revenues" rows are
  subtracted from it (that money isn't new);
- rows under "Additional investments" have no heading in the text: each becomes a measure of its own, with the
  table's one-line description as its text and no words (it isn't in the running text);
- a few adjustments belong to a whole section or chapter, not to the row above them; `manual/cost_matches.csv` says
  where each goes (and why), and any row the matching can't place goes there too;
- **check:** the rows of every chapter add up to the chapter's net fiscal impact (`checks/reconciliation.csv`).

**Words.** The words of the measure's own running text (paragraphs and bullet points), headings excluded.

Output: `data/measures.csv` (one row per measure, with its text), `data/cost_rows.csv` (every cost row and the measure
it went to), and the checks in `checks/`.

In [1]:
import csv
import os
import re
from difflib import SequenceMatcher

import pandas as pd
from bs4 import BeautifulSoup

YEARS = [2021, 2022, 2023, 2024, 2025, 2026]
RAW, DATA, CHECKS, MANUAL = "raw", "data", "checks", "manual"
for d in (DATA, CHECKS):
    os.makedirs(d, exist_ok=True)

BASE_LEVEL = {2021: 4}            # heading level of a measure: h4 in Budget 2021, h3 in the others
MIN_WORDS = 30                    # an uncosted heading needs at least this many words to count as a measure
MATCH_OK, MATCH_FUZZY = 0.85, 0.72  # name similarity: accepted / accepted but listed for review

## Text helpers
- `clean`: one space between words, plain quotes, and no "View the impact assessment" links (Budget 2022 puts them
  inside its headings and table rows).
- `strip_notes`: removes footnote marks at the end of a table row ("Powering Canada Strong*", "... Claims 1",
  "... businesses 1, 2,*").
- `norm`: the form used to compare a table row with a heading (lower case, letters and digits only, list numbers
  like "1." removed).

In [2]:
LINK = re.compile(r"View (the )?(impact assessment|summary)( for [^.]*)?\.?", re.I)
FOOTNOTE = re.compile(r"(\s*[*¹²³⁰-⁹]+|\s+\d{1,2}(\s*,\s*\d{1,2})*\s*,?\s*\*?)+$")
SECNUM = re.compile(r"^\d+\.\d+(\.\d+)*\.?\s")                         # a numbered section: "1.2 ", "2.1. "


def clean(s):
    s = s.replace("\xa0", " ").replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    s = LINK.sub(" ", re.sub(r"\s+", " ", s))
    return re.sub(r"\s+", " ", s).strip()


def strip_notes(s):
    return FOOTNOTE.sub("", s).strip()


def norm(s):
    s = strip_notes(clean(s)).lower().replace("&", " and ")
    s = re.sub(r"^\d+(\.\d+)*\.?\s+", "", s)                             # "1. Support for ..." -> "support for ..."
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", s)).strip()


def words(text):
    return len(re.findall(r"[A-Za-z]+", text))


def number(s):
    """A cost cell in millions: '1,375' -> 1375, '- 59,569' -> -59569, '…' or '-' (no change) -> 0."""
    s = clean(s).replace(",", "").replace(" ", "").replace("−", "-").replace("–", "-").replace("—", "-")
    s = s.rstrip("*")
    if s in ("-", "…", "...", "…."):
        return 0.0
    return float(s) if re.fullmatch(r"-?\d+(\.\d+)?", s) else None

## 1. The cost tables
Each row of a chapter's table is one of:
- a **section** row ("1.1 Building More Homes"), or "Additional investments / actions – ..." (rows below it have no
  heading in the text), or a sub-total;
- a **measure** row;
- an **adjustment** row ("Less: ...", "Forgone revenues", "Year-over-year reallocation", "Administrative costs",
  "Realignment of existing funding"): added to (or subtracted from) the measure row just above it, unless it follows a sub-total or is one of the known section-level adjustments, in which case it
  stays on its own and `manual/cost_matches.csv` says where it goes;
- a **description** (one cell, under an "Additional investments" row), a note, or the chapter's **net fiscal impact**
  (the total, used as the check; anything after it is a note).

In [3]:
SECTION_ROW = re.compile(r"^(\d+\.\d+|additional (investments|actions|measures))|sub-?total", re.I)
SUBTOTAL = re.compile(r"sub-?total", re.I)
NFI = re.compile(r"(net )?fiscal impact\s*$", re.I)                  # "Chapter 2 - Net Fiscal Impact" (2022: "Fiscal Impact")
ADJUST = re.compile(r"^(less\s*:|plus\s*:|forgone revenues|year-over-\s?year reallocation|administrative costs$|"
                    r"realignment of existing funding)", re.I)
WHOLE_SECTION = re.compile(r"anticipated cabinet decisions|provisioned in budget \d{4} for future reviews|unrealized budget \d{4}", re.I)


def is_cost_table(t):
    cap = clean(t.find("caption").get_text(" ")) if t.find("caption") else ""
    first = [tr.find(["th", "td"]) for tr in t.find_all("tr")]
    return re.match(r"^(Chapter|Part)\s*\d", cap) and any(c is not None and NFI.search(clean(c.get_text(" "))) for c in first)


def read_table(t, year, page):
    cap = clean(t.find("caption").get_text(" "))
    table = re.sub(r"\s*millions of dollars.*$", "", cap, flags=re.I)
    items, nfi, section, prev = [], None, "", None
    for tr in t.find_all("tr"):
        for sup in tr.find_all("sup"):
            sup.decompose()                                              # footnote marks
        cells = tr.find_all(["th", "td"])
        if not cells:
            continue
        label = clean(cells[0].get_text(" "))
        total = number(cells[-1].get_text(" ")) if len(cells) > 1 else None
        if NFI.search(label):
            nfi = total
            break                                                        # what follows is "of which" and notes
        if total is None:                                                # one cell: a description, a note, the header
            if prev == "item" and items and not items[-1]["description"] and not label.lower().startswith("note"):
                items[-1]["description"] = label
            continue
        if SECTION_ROW.search(label):
            if SUBTOTAL.search(label):
                prev = "subtotal"
            else:
                section, prev = strip_notes(re.sub(r"^\d+(\.\d+)*\.?\s*", "", label)), "section"
            continue
        name = strip_notes(label)
        if ADJUST.match(label) and prev == "item" and not WHOLE_SECTION.search(label):
            items[-1]["net"] += total                                    # nets into the measure row above
            items[-1]["adjustments"].append(f"{name} ({total:+,.0f})")
            continue
        kind = "adjustment" if ADJUST.match(label) else ("table" if section.lower().startswith("additional") else "item")
        items.append({"year": year, "page": page, "table": table, "section": section, "row": name, "kind": kind,
                      "gross": total, "net": total, "adjustments": [], "description": ""})
        prev = "item" if kind != "adjustment" else "adjustment"
    return items, nfi


cost_items, recon = [], []
for year in YEARS:
    for page in sorted(os.listdir(os.path.join(RAW, str(year)))):
        if not re.match(r"^(chap|p)\d+-en\.html$", page):
            continue
        main = BeautifulSoup(open(os.path.join(RAW, str(year), page), "rb").read(), "html.parser").find("main")
        for t in main.find_all("table"):
            if not is_cost_table(t):
                continue
            items, nfi = read_table(t, year, page)
            for i, it in enumerate(items):
                it["order"] = len(cost_items) + i
            cost_items += items
            recon.append({"year": year, "page": page, "table": items[0]["table"] if items else "", "net_fiscal_impact": nfi,
                          "sum_of_rows": round(sum(it["net"] for it in items), 1)})
recon = pd.DataFrame(recon)
pages = [(y, p) for y in YEARS for p in os.listdir(os.path.join(RAW, str(y))) if re.match(r"^(chap|p)\d+-en\.html$", p)]
no_table = [(y, p) for y, p in pages if not ((recon["year"] == y) & (recon["page"] == p)).any()]
assert not no_table, f"chapter pages without a cost table: {no_table}"
recon["difference"] = (recon["sum_of_rows"] - recon["net_fiscal_impact"]).round(1)
print(f"{len(cost_items)} cost rows in {len(recon)} chapter tables, at least one on each of the {len(pages)} chapter pages")
print("largest gap between a table's rows and its net fiscal impact: "
      f"${recon['difference'].abs().max():,.0f} million (rounding in the published tables)")
assert recon["difference"].abs().max() <= 10, recon[recon["difference"].abs() > 10]

1104 cost rows in 44 chapter tables, at least one on each of the 38 chapter pages
largest gap between a table's rows and its net fiscal impact: $5 million (rounding in the published tables)


## 2. The headings and their text
For each page: every heading (h1 to h5) in order, with the paragraphs and bullet points that follow it, and whether it
sits in a call-out box. Removed first: tables, charts (figures), collapsible "text versions", scripts, navigation,
impact reports, and everything from "Page details" on.

In [4]:
BOX = re.compile(r"^(well|panel|panel-.+|cc|cc-.+|alert|alert-.+)$")
GENERIC = re.compile(r"^(on this page|key ongoing actions|ongoing actions|impacts? report|what we heard|thank you for your help|"
                     r"key investments|budget \d{4} (investments|measures)|text version)", re.I)
FIGURE = re.compile(r"^(figure|chart|table|graph)\s*\d", re.I)              # a chart's title: not a heading of the text


def in_box(el, main):
    p = el.parent
    while p is not None and p is not main:
        if p.name in ("aside",) or any(BOX.match(c) for c in p.get("class", [])):
            return True
        p = p.parent
    return False


def read_headings(main):
    for tag in main.find_all(["table", "script", "style", "nav", "figure", "details", "noscript"]):
        tag.decompose()
    reports = [h for h in main.find_all(["h2", "h3", "h4", "h5"]) if re.match(r"^impacts? report", clean(h.get_text(" ")), re.I)]
    for h in reports:                                                    # impact reports: drop the whole panel
        if getattr(h, "decomposed", False):
            continue
        box = h.find_parent(lambda tag: any("panel" in c for c in tag.get("class", [])))
        (box or h).decompose()
    nodes = []
    for el in main.find_all(["h1", "h2", "h3", "h4", "h5", "h6", "p", "li"]):
        if el.name[0] == "h":
            title = clean(el.get_text(" "))
            if title.lower().startswith("page details"):
                break
            if FIGURE.match(title):
                continue                                                 # text after a chart title stays with the heading above
            nodes.append({"i": len(nodes), "level": int(el.name[1]), "title": title, "box": in_box(el, main),
                          "generic": bool(GENERIC.match(title)), "own": []})
        elif nodes and el.find_parent("li") is None and (el.name == "li" or el.find_parent("li") is None):
            txt = clean(el.get_text(" "))
            if txt:
                nodes[-1]["own"].append(txt)
    return nodes

## 3. Match each cost row to a heading
A row usually repeats its measure's heading word for word. Otherwise:
- "Heading – part" rows (e.g. "Protecting Workers ... - Support for Forestry") go to the part's own heading if it has
  one under the measure (or, word for word, elsewhere on the page), else to the measure;
- a heading with a colon also matches its first half ("Mackenzie Valley Highway: connecting communities ...");
- a row that repeats the first words of a heading (at least three) matches it ("Expanding the Low Carbon Economy
  Fund" → "Expanding the Low Carbon Economy Fund and Supporting Clean Energy in Yukon");
- names that are close but not identical (similarity of at least 0.85) are accepted; between 0.72 and 0.85 accepted
  but listed in `checks/matching.csv`; below that, the row has no heading.
Where several headings tie, a heading outside a box, then one at the measure level, wins.
`manual/cost_matches.csv` overrides the automatic match for the rows it lists.

In [5]:
manual = {(int(r["year"]), r["page"], r["row"]): r for r in csv.DictReader(open(os.path.join(MANUAL, "cost_matches.csv"), encoding="utf-8"))}
SPLIT = re.compile(r"\s+[–—-]\s+")


def forms(title):
    f = {norm(title)}
    if ":" in title:
        f.add(norm(title.split(":")[0]))
    return f


def score(name, node):
    n, out = norm(name), 0.0
    for f in forms(node["title"]):
        if n == f:
            return 1.0
        s = SequenceMatcher(None, n, f).ratio()
        if len(n.split()) >= 3 and (f + " ").startswith(n + " "):
            s = max(s, 0.88)                     # the row repeats the start of the heading ("Expanding the Low Carbon Economy Fund")
        out = max(out, s)
    return out


def best(name, cands, base):
    if not cands:
        return None, 0.0
    ranked = sorted(cands, key=lambda k: (-round(score(name, k[1]), 4), k[1]["box"], abs(k[1]["level"] - base), k[0]))
    return ranked[0][0], score(name, ranked[0][1])


def subtree(nodes, i):
    out = []
    for j in range(i + 1, len(nodes)):
        if nodes[j]["level"] <= nodes[i]["level"] and not nodes[j]["box"]:
            break
        out.append(j)
    return out


def match(item, nodes, base):
    cands = [(i, n) for i, n in enumerate(nodes) if not n["generic"] and n["level"] >= 2]
    m = manual.get((item["year"], item["page"], item["row"]))
    if m is not None:
        if m["action"] == "heading":
            hits = [i for i, n in cands if n["title"] == m["target"]]
            assert hits, f"manual/cost_matches.csv: heading not found: {m['target']!r}"
            return hits[0], 1.0, "manual"
        return None, 0.0, "manual: " + m["action"]
    if item["kind"] == "adjustment":
        return None, 0.0, "none"
    name = item["row"]
    parts = SPLIT.split(name, maxsplit=1)
    if len(parts) == 2:
        ix, sx = best(parts[0], cands, base)
        if sx >= MATCH_OK:
            sub = [(j, nodes[j]) for j in subtree(nodes, ix) if not nodes[j]["generic"]]
            iy, sy = best(parts[1], sub, base)
            if sy >= MATCH_OK:
                return iy, sy, "part"
            iy, sy = best(parts[1], cands, base)                          # the part's own heading elsewhere on the page
            if sy == 1.0 and len(norm(parts[1]).split()) >= 2:
                return iy, sy, "part (elsewhere)"
            return ix, sx, "heading - part"
        i_full, s_full = best(name, cands, base)
        if s_full >= MATCH_OK:
            return i_full, s_full, "close" if s_full < 1 else "exact"
        iy, sy = best(parts[1], cands, base)
        if sy >= MATCH_OK:
            return iy, sy, "part"
    i, s = best(name, cands, base)
    if s >= MATCH_OK:
        return i, s, "exact" if s == 1 else "close"
    if s >= MATCH_FUZZY:
        return i, s, "fuzzy"
    return None, s, "none"

## 4. Build the measures
Walk down each page's headings, keeping track of which heading each one sits under (a box heading always sits under
the heading before it). Each heading is one of:
- **measure**: it has a cost row, or it is at the measure level (outside a box);
- **section**: numbered ("1.2 ..."), or above the measure level; its introduction is left out;
- **left out**: impact reports, "key ongoing actions", "what we heard", "past investments since 2015"... (and what
  sits under them);
- **part**: anything else; its text joins the nearest measure above it.

In [6]:
measures, rows_out, dropped = [], [], []
for year in YEARS:
    base = BASE_LEVEL.get(year, 3)
    for page in sorted(os.listdir(os.path.join(RAW, str(year))), key=lambda p: int(re.search(r"\d+", p).group()) if re.search(r"\d", p) else 0):
        if not re.match(r"^(chap|p)\d+-en\.html$", page):
            continue
        main = BeautifulSoup(open(os.path.join(RAW, str(year), page), "rb").read(), "html.parser").find("main")
        h1 = clean(main.find("h1").get_text(" ")) if main.find("h1") else page
        nodes = read_headings(main)
        items = [it for it in cost_items if it["year"] == year and it["page"] == page]
        costs = {}
        for it in items:
            i, s, how = match(it, nodes, base)
            it["match"], it["score"], it["node"] = how, round(s, 3), i
            if i is not None:
                costs.setdefault(i, []).append(it)

        # roles and nesting
        stack, chapter, owner = [], re.sub(r"^(Archived\s*-\s*)?((Chapter|Part)\s*\d+\s*:?\s*)?", "", h1), {}
        for i, n in enumerate(nodes):
            if n["level"] == 1:
                continue
            if year == 2021 and re.match(r"^Chapter\s*\d+", n["title"]):
                chapter = re.sub(r"^Chapter\s*\d+\s*:?\s*", "", n["title"])
            eff = 6 + n["level"] if n["box"] else n["level"]
            while stack and stack[-1][0] >= eff:
                stack.pop()
            if i in costs:
                role = "measure"
            elif n["generic"]:
                role = "out"
            elif not n["box"] and (SECNUM.match(n["title"]) or n["level"] < base):
                role = "section"
            elif not n["box"] and n["level"] == base:
                role = "measure"
            else:
                role = "part"
            n["role"], n["chapter"] = role, chapter
            n["section"] = next((re.sub(r"^\d+(\.\d+)*\.?\s*", "", stack[k][1]["title"]) for k in range(len(stack) - 1, -1, -1)
                                 if stack[k][1]["role"] == "section" and not re.match(r"^Chapter\s*\d", stack[k][1]["title"])), "")
            stack.append((eff, n))
            if role == "measure":
                owner[i] = i
            elif role == "part":                                         # text goes to the nearest measure above
                for _, up in reversed(stack[:-1]):
                    if up["role"] == "measure":
                        owner[i] = up["i"]
                        break
                    if up["role"] in ("section", "out"):
                        break
            if role == "part" and i not in owner and n["own"]:
                dropped.append({"year": year, "page": page, "heading": n["title"], "why": "outside any measure",
                                "words": words(" ".join(n["own"]))})
            if role == "out" and n["own"]:
                dropped.append({"year": year, "page": page, "heading": n["title"], "why": "left out (generic box)",
                                "words": words(" ".join(n["own"]))})

        texts = {i: [] for i in owner.values()}
        for i, o in owner.items():
            texts[o] += nodes[i]["own"]
        seen = {}
        for i in sorted(texts):
            n, its = nodes[i], costs.get(i, [])
            text = " ".join(texts[i])
            if not its and words(text) < MIN_WORDS:
                dropped.append({"year": year, "page": page, "heading": n["title"], "why": f"fewer than {MIN_WORDS} words, no cost",
                                "words": words(text)})
                continue
            heading = strip_notes(n["title"]) if n["box"] else n["title"]
            heading = re.sub(r"^\d+(\.\d+)*\.?\s+", "", heading)          # "1. Support for Forestry" -> "Support for Forestry"
            seen[heading] = seen.get(heading, 0) + 1
            mid = f"{year}|{page}|{heading}" + (f" #{seen[heading]}" if seen[heading] > 1 else "")
            for it in its:
                it["measure_id"] = mid
            measures.append({"id": mid, "year": year, "page": page, "chapter": n["chapter"], "section": n["section"],
                             "heading": heading, "level": n["level"], "box": n["box"], "source": "text",
                             "money_m": sum(it["net"] for it in its) if its else None,
                             "gross_m": sum(it["gross"] for it in its) if its else None, "cost_rows": len(its),
                             "words": words(text), "text": text})

        # rows without a heading: "Additional investments" (and any row the matching couldn't place) are measures of
        # their own, with the table's description as text
        for it in items:
            if it["node"] is not None or it["match"].startswith("manual: spread"):
                continue                                                 # placed under a heading, or spread later
            if it["kind"] == "adjustment" and it["match"] != "manual: table":
                continue                                                 # an adjustment nobody placed: caught below
            seen[it["row"]] = seen.get(it["row"], 0) + 1
            mid = f"{year}|{page}|{it['row']}" + (f" #{seen[it['row']]}" if seen[it["row"]] > 1 else "")
            it["measure_id"] = mid
            measures.append({"id": mid, "year": year, "page": page, "chapter": chapter, "section": it["section"],
                             "heading": it["row"], "level": None, "box": False, "source": "table",
                             "money_m": it["net"], "gross_m": it["gross"], "cost_rows": 1, "words": 0,
                             "text": it["description"]})

print(f"{len(measures)} measures; {sum(m['money_m'] is not None for m in measures)} with a cost; "
      f"{sum(m['source'] == 'table' for m in measures)} only in the cost tables")

1317 measures; 1051 with a cost; 278 only in the cost tables


## 5. Adjustments that belong to a section or a whole chapter
`manual/cost_matches.csv` says where each goes:
- `spread_section_savings`: shared among the savings rows of its section, in proportion to their size (e.g. Budget
  2026's savings in section 6.3 had already been booked in Budget 2025 as "future reviews", so they aren't new);
- `spread_chapter_spending`: shared among the chapter's spending rows in proportion (Budget 2021's "provisions for
  anticipated Cabinet decisions": money already set aside in earlier budgets for decisions not yet made);
- `heading`: added to one measure (or, for a measure row whose wording differs from its heading, matched to it);
- `table`: kept as a measure of its own.

In [7]:
by_id = {m["id"]: m for m in measures}
for it in cost_items:
    action = it["match"].replace("manual: ", "") if it["match"].startswith("manual: ") else None
    if action is None or action == "table":
        continue                                                         # "table": already its own measure
    same = [x for x in cost_items if x["year"] == it["year"] and x["page"] == it["page"] and x["table"] == it["table"]
            and x is not it and x.get("measure_id")]
    if action == "spread_section_savings":
        pool = [x for x in same if x["section"] == it["section"] and x["net"] < 0]
    elif action == "spread_chapter_spending":
        pool = [x for x in same if x["net"] > 0]
    else:
        raise ValueError(f"unknown action {action!r} in manual/cost_matches.csv")
    tot = sum(x["net"] for x in pool)
    for x in pool:
        share = it["net"] * x["net"] / tot
        by_id[x["measure_id"]]["money_m"] += share
    it["measure_id"] = f"(spread over {len(pool)} rows)"
    print(f"  {it['year']} {it['page']}: {it['row'][:70]} ({it['net']:+,.0f}) spread over {len(pool)} rows ({action})")

  2021 p2-en.html: Less: Provisions for Anticipated Cabinet Decisions Not Yet Made Includ (-9,083) spread over 16 rows (spread_chapter_spending)
  2021 p3-en.html: Less: Provisions for Anticipated Cabinet Decisions Not Yet Made Includ (-2,188) spread over 25 rows (spread_chapter_spending)
  2026 chap6-en.html: Less: amounts provisioned in Budget 2025 for future reviews (+2,064) spread over 5 rows (spread_section_savings)


## 6. Checks
- every cost row is accounted for: each chapter's rows went to measures (from the text or the table), or were spread;
- which rows matched a heading only approximately, and which headings were left out.

In [8]:
cr = pd.DataFrame(cost_items)
cr["adjustments"] = cr["adjustments"].apply(lambda a: "; ".join(a))
cr = cr[["year", "page", "table", "section", "row", "kind", "gross", "adjustments", "net", "description", "match", "score",
         "measure_id"]].rename(columns={"gross": "gross_m", "net": "net_m"})
cr.to_csv(os.path.join(DATA, "cost_rows.csv"), index=False)
unplaced = cr[cr["measure_id"].isna()]
assert unplaced.empty, f"cost rows not placed anywhere:\n{unplaced[['year', 'page', 'row', 'net_m']]}"

M = pd.DataFrame(measures)
M["money_b"] = (M["money_m"] / 1000).round(4)
M["gross_b"] = (M["gross_m"] / 1000).round(4)
M = M.drop(columns=["money_m", "gross_m"])
M = M[["id", "year", "page", "chapter", "section", "heading", "level", "box", "source", "money_b", "gross_b", "cost_rows",
       "words", "text"]]
assert M["id"].is_unique
M.to_csv(os.path.join(DATA, "measures.csv"), index=False)

src = dict(zip(M["id"], M["source"]))
cr["goes_to"] = cr["measure_id"].map(src).fillna("spread")
for part in ("text", "table", "spread"):
    recon["to_" + part + "_measures" if part != "spread" else "spread_over_rows"] = [
        round(cr[(cr["year"] == r.year) & (cr["page"] == r.page) & (cr["table"] == r.table) & (cr["goes_to"] == part)]["net_m"].sum(), 1)
        for r in recon.itertuples()]
recon.to_csv(os.path.join(CHECKS, "reconciliation.csv"), index=False)
review = cr[cr["match"].isin(["fuzzy", "close", "part", "part (elsewhere)", "heading - part"]) | (cr["kind"] == "item") & cr["match"].eq("none")]
review.to_csv(os.path.join(CHECKS, "matching.csv"), index=False)
pd.DataFrame(dropped).to_csv(os.path.join(CHECKS, "left_out.csv"), index=False)

summary = M.groupby("year").agg(measures=("id", "size"), from_text=("source", lambda s: (s == "text").sum()),
                                with_cost=("money_b", lambda s: s.notna().sum()), money_b=("money_b", "sum"),
                                words=("words", "sum"))
summary["net_fiscal_impact_b"] = recon.groupby("year")["net_fiscal_impact"].sum() / 1000
print(summary.round(1).to_string())
print(f"\nrows matched: {(cr['match'] == 'exact').sum()} exactly, {cr['match'].isin(['close', 'part', 'part (elsewhere)', 'heading - part']).sum()} "
      f"closely or by part, {(cr['match'] == 'fuzzy').sum()} approximately (see checks/matching.csv), "
      f"{cr['match'].str.startswith('manual').sum()} by hand, {((cr['match'] == 'none') & (cr['kind'] == 'item')).sum()} "
      f"measure rows with no heading (kept as table-only measures)")
print(f"{len(dropped)} headings left out (checks/left_out.csv), {sum(d['words'] for d in dropped):,} words")

      measures  from_text  with_cost  money_b  words  net_fiscal_impact_b
year                                                                     
2021       318        267        277    142.3  54684                142.3
2022       203        184        159     37.5  33243                 37.5
2023       209        144        161     45.8  28448                 45.8
2024       292        228        227     34.8  67357                 34.8
2025       118         95         86     89.7  29593                 89.7
2026       177        121        141    142.0  50525                142.0

rows matched: 726 exactly, 71 closely or by part, 4 approximately (see checks/matching.csv), 27 by hand, 0 measure rows with no heading (kept as table-only measures)
79 headings left out (checks/left_out.csv), 18,502 words
